# Poverty Hotspot Identifier - Interactive Notebook

This notebook demonstrates how to use the `PovertyHotspotModel` class for analyzing poverty patterns in Nigeria.

## Setup

In [ ]:
# Import required libraries
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path
sys.path.insert(0, 'src')

from poverty_hotspot_model import PovertyHotspotModel

# Configure visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## Step 1: Initialize the Model

In [ ]:
# Create model instance
model = PovertyHotspotModel()
print("Model initialized successfully!")

## Step 2: Load and Clean Data

This step will:
- Load the CSV file
- Impute missing values using KNN
- Standardize features

In [ ]:
# Load and preprocess data
data_file = 'nigeria_mpi_lga_data.csv'
cleaned_data = model.load_and_clean_data(data_file)

# Display first few rows
print("\nCleaned Data Sample:")
cleaned_data.head()

## Step 3: Dimensionality Reduction with PCA

Apply PCA to reduce the feature space while retaining most variance.

In [ ]:
# Apply PCA - keep components explaining 95% variance
pca_data = model.reduce_dimensionality(n_components=0.95)

print(f"\nOriginal features: {len(model.feature_columns)}")
print(f"PCA components: {model.pca.n_components_}")
print(f"Variance explained: {model.pca.explained_variance_ratio_.sum():.4f}")

## Step 4: Find Optimal Number of Clusters

Use Elbow Method and Silhouette Score to determine optimal k.

In [ ]:
# Find optimal clusters
metrics = model.find_optimal_clusters(k_range=range(2, 11))

# Display metrics
print("\nCluster Optimization Metrics:")
for k, (wcss, sil) in enumerate(zip(metrics['wcss'], metrics['silhouette_scores']), start=2):
    print(f"k={k}: WCSS={wcss:.2f}, Silhouette={sil:.4f}")

## Step 5: Train Clustering Models

Train both K-Means and DBSCAN models.

In [ ]:
# Train models with optimal k (adjust based on plots above)
k_optimal = 4  # Modify based on your analysis

labels = model.train_clustering_models(
    k_optimal=k_optimal,
    dbscan_eps=0.5,
    dbscan_min_samples=5,
    use_kmeans=True
)

print(f"\nK-Means cluster distribution:")
print(pd.Series(labels['kmeans_labels']).value_counts().sort_index())

## Step 6: Generate and Analyze Cluster Profiles

Understand what characterizes each cluster.

In [ ]:
# Generate cluster profiles
profiles = model.generate_cluster_profiles()

# Display profiles
print("\nCluster Profiles:")
profiles

### Visualize Cluster Profiles

In [ ]:
# Create heatmap of cluster profiles
plt.figure(figsize=(14, 6))
sns.heatmap(profiles.T, annot=True, fmt='.2f', cmap='RdYlGn', center=profiles.values.mean())
plt.title('Cluster Profiles Heatmap', fontsize=16, fontweight='bold')
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

## Step 7: Save Results

Save the processed data and trained models.

In [ ]:
# Save results
paths = model.save_results(
    output_csv='processed_hotspots.csv',
    model_dir='models'
)

print("\nSaved files:")
print(f"CSV: {paths['csv_path']}")
for name, path in paths['model_paths'].items():
    print(f"{name}: {path}")

## Step 8: Inference on New Data

Demonstrate how to use trained models for predictions.

In [ ]:
# Create sample new LGA data
new_lgas = pd.DataFrame({
    'state': ['TestState'] * 2,
    'lga': ['TestLGA_1', 'TestLGA_2'],
    'lga_id': [9999, 10000],
    'nutrition_score': [55, 80],
    'food_insecurity_rate': [40, 15],
    'years_of_schooling': [5, 10],
    'school_attendance_rate': [60, 90],
    'unemployment_rate': [30, 10],
    'security_shock_incidence': [20, 5],
    'electricity_access': [35, 85],
    'sanitation_access': [45, 75],
    'water_reliability': [40, 70],
    'housing_quality_index': [50, 85],
    'nightlight_intensity': [10, 40],
    'population_density': [500, 3000],
    'road_density': [0.8, 4.0],
})

# Predict clusters
predictions = model.predict_cluster(new_lgas)

print("\nPredictions for new LGAs:")
new_lgas['predicted_cluster'] = predictions
new_lgas[['lga', 'food_insecurity_rate', 'electricity_access', 'predicted_cluster']]

## Additional Analysis

### Cluster Distribution by State

In [ ]:
# Load processed data
results = pd.read_csv('processed_hotspots.csv')

# Cluster distribution by state
cluster_dist = pd.crosstab(results['state'], results['cluster_label'])

# Plot
cluster_dist.plot(kind='bar', stacked=True, figsize=(14, 6))
plt.title('Cluster Distribution by State', fontsize=16, fontweight='bold')
plt.xlabel('State', fontsize=12)
plt.ylabel('Number of LGAs', fontsize=12)
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Feature Importance in Clustering

In [ ]:
# PCA component loadings
loadings = pd.DataFrame(
    model.pca.components_.T,
    columns=[f'PC{i+1}' for i in range(model.pca.n_components_)],
    index=model.feature_columns
)

# Plot loadings for first 2 components
plt.figure(figsize=(12, 8))
loadings[['PC1', 'PC2']].plot(kind='barh', figsize=(10, 8))
plt.title('Feature Loadings on First Two Principal Components', fontsize=16, fontweight='bold')
plt.xlabel('Loading', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.axvline(x=0, color='k', linestyle='--', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the complete poverty hotspot analysis pipeline:

1. **Data Loading & Preprocessing**: KNN imputation + standardization
2. **Dimensionality Reduction**: PCA for composite poverty indices
3. **Cluster Optimization**: Elbow method + silhouette analysis
4. **Model Training**: K-Means and DBSCAN clustering
5. **Cluster Profiling**: Understanding poverty patterns
6. **Results Export**: Saving data and models
7. **Inference**: Predicting clusters for new LGAs

Next steps:
- Integrate with GIS tools for spatial visualization
- Design targeted interventions based on cluster profiles
- Update models with new data periodically